# 15. Agent 并行工具调用：怎样用 call_id 防止结果串线？

## 面试回答主线

并行工具调用的核心不是 `gather`，而是为每个计划动作分配稳定且唯一的 call_id。工具结果必须回传原 call_id，编排器只能按 ID 关联，绝不能按完成顺序与计划顺序 zip。因为网络延迟不同，按顺序关联会把库存数字当成天气，把风控结论塞给支付调用，最终生成语法通顺但事实错误的回答。一个可靠状态机还应拒绝重复 ID、未知 ID、缺失结果和重复提交，并给每次调用设置超时与幂等键。面试时我会用至少五个双工具任务复现串线，再手写稳定 ID、乱序执行和 keyed join。生产环境还要补取消、重试、trace、schema 校验和 exactly-once effect，而不是只保证一次函数返回。

## 1. 真实案例：六个必须并行查询两个来源的 Agent 任务

每个用户问题都拆成两个工具调用，且特意设置不同延迟让完成顺序与计划顺序相反。字段包含任务 ID、自然语言问题、工具名、参数和模拟延迟；工具返回的内容来自小型业务数据表。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示并行任务输入
tasks = [{"task_id": "A01", "question": "北京有雨且雨伞有货吗？", "calls": [("weather", "北京", 80), ("inventory", "雨伞", 20)]}, {"task_id": "A02", "question": "上海炎热时空调还能买吗？", "calls": [("weather", "上海", 70), ("inventory", "空调", 10)]}, {"task_id": "A03", "question": "订单 O9 是否已支付并可以发货？", "calls": [("payment", "O9", 90), ("shipment", "O9", 30)]}, {"task_id": "A04", "question": "用户 U2 风险可控且额度足够吗？", "calls": [("risk", "U2", 75), ("credit", "U2", 25)]}, {"task_id": "A05", "question": "深圳天气适合且会议室空闲吗？", "calls": [("weather", "深圳", 60), ("room", "深圳", 15)]}, {"task_id": "A06", "question": "文档 D7 有权限且仍是最新版吗？", "calls": [("permission", "D7", 85), ("version", "D7", 35)]}]  # 定义六个具有真实语义的双工具任务
tool_data = {("weather", "北京"): "小雨", ("inventory", "雨伞"): "库存 18", ("weather", "上海"): "33℃", ("inventory", "空调"): "库存 4", ("payment", "O9"): "已支付", ("shipment", "O9"): "待出库", ("risk", "U2"): "低风险", ("credit", "U2"): "额度 5000", ("weather", "深圳"): "晴", ("room", "深圳"): "B201 空闲", ("permission", "D7"): "允许读取", ("version", "D7"): "v12 最新"}  # 构造可复现的工具侧业务数据
preview = [{"任务": task["task_id"], "问题": task["question"], "计划工具": [call[0] for call in task["calls"]], "延迟毫秒": [call[2] for call in task["calls"]]} for task in tasks]  # 提取计划顺序与延迟供学习者观察
print("并行工具任务输入预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示计划顺序为何会与完成顺序不同

并行工具任务输入预览：
[{'任务': 'A01',
  '问题': '北京有雨且雨伞有货吗？',
  '计划工具': ['weather', 'inventory'],
  '延迟毫秒': [80, 20]},
 {'任务': 'A02',
  '问题': '上海炎热时空调还能买吗？',
  '计划工具': ['weather', 'inventory'],
  '延迟毫秒': [70, 10]},
 {'任务': 'A03',
  '问题': '订单 O9 是否已支付并可以发货？',
  '计划工具': ['payment', 'shipment'],
  '延迟毫秒': [90, 30]},
 {'任务': 'A04',
  '问题': '用户 U2 风险可控且额度足够吗？',
  '计划工具': ['risk', 'credit'],
  '延迟毫秒': [75, 25]},
 {'任务': 'A05',
  '问题': '深圳天气适合且会议室空闲吗？',
  '计划工具': ['weather', 'room'],
  '延迟毫秒': [60, 15]},
 {'任务': 'A06',
  '问题': '文档 D7 有权限且仍是最新版吗？',
  '计划工具': ['permission', 'version'],
  '延迟毫秒': [85, 35]}]


## 2. Baseline（基线）：把完成结果按顺序 zip 回计划

下面先模拟并发：延迟小的结果先完成。错误基线丢掉 call_id，然后把第一条完成结果交给第一个计划调用。六个任务都会出现工具名与结果来源不一致，这种错误很隐蔽，因为最终文本仍然像一句自然语言。

In [2]:
def execute_without_id(call):  # 模拟一个不回传调用标识的工具执行器
    tool, argument, latency = call  # 读取计划中的工具、参数与模拟延迟
    return {"source_tool": tool, "value": tool_data[(tool, argument)], "latency": latency}  # 返回业务结果但故意遗漏 call_id
baseline_rows = []  # 收集按完成顺序错误关联的结果
for task in tasks:  # 逐个执行双工具任务
    completed = sorted((execute_without_id(call) for call in task["calls"]), key=lambda result: result["latency"])  # 用延迟排序模拟并行完成顺序
    wrongly_joined = []  # 保存计划槽位与实际来源的错误配对
    for planned, result in zip(task["calls"], completed):  # 错误地假设完成顺序等于计划顺序
        wrongly_joined.append({"计划工具": planned[0], "实际来源": result["source_tool"], "值": result["value"]})  # 记录串线后的可观察内容
    mismatch_count = sum(item["计划工具"] != item["实际来源"] for item in wrongly_joined)  # 统计当前任务有多少结果被放错槽位
    baseline_rows.append({"任务": task["task_id"], "错误关联": wrongly_joined, "串线数": mismatch_count})  # 保存逐任务基线错误
print("按完成顺序关联的错误基线：")  # 标注当前输出属于错误方案
pprint(baseline_rows, sort_dicts=False)  # 展示库存、天气等结果如何发生串线

按完成顺序关联的错误基线：
[{'任务': 'A01',
  '错误关联': [{'计划工具': 'weather', '实际来源': 'inventory', '值': '库存 18'},
           {'计划工具': 'inventory', '实际来源': 'weather', '值': '小雨'}],
  '串线数': 2},
 {'任务': 'A02',
  '错误关联': [{'计划工具': 'weather', '实际来源': 'inventory', '值': '库存 4'},
           {'计划工具': 'inventory', '实际来源': 'weather', '值': '33℃'}],
  '串线数': 2},
 {'任务': 'A03',
  '错误关联': [{'计划工具': 'payment', '实际来源': 'shipment', '值': '待出库'},
           {'计划工具': 'shipment', '实际来源': 'payment', '值': '已支付'}],
  '串线数': 2},
 {'任务': 'A04',
  '错误关联': [{'计划工具': 'risk', '实际来源': 'credit', '值': '额度 5000'},
           {'计划工具': 'credit', '实际来源': 'risk', '值': '低风险'}],
  '串线数': 2},
 {'任务': 'A05',
  '错误关联': [{'计划工具': 'weather', '实际来源': 'room', '值': 'B201 空闲'},
           {'计划工具': 'room', '实际来源': 'weather', '值': '晴'}],
  '串线数': 2},
 {'任务': 'A06',
  '错误关联': [{'计划工具': 'permission', '实际来源': 'version', '值': 'v12 最新'},
           {'计划工具': 'version', '实际来源': 'permission', '值': '允许读取'}],
  '串线数': 2}]


## 3. 手写核心算法：稳定 call_id 与乱序结果信封

call_id 在规划阶段产生，并随着请求穿过执行器、重试器和结果队列。这里用 `任务 ID + 计划序号` 生成稳定 ID；真实系统可用 UUID，但重试必须复用同一业务调用的 ID。结果信封保留 tool、argument 和 call_id，便于 schema 与可观测性检查。

In [3]:
def plan_calls(task):  # 为一个 Agent 任务建立带稳定标识的调用计划
    planned = []  # 初始化有序调用计划
    for index, raw_call in enumerate(task["calls"], start=1):  # 按规划顺序遍历工具动作
        tool, argument, latency = raw_call  # 拆出工具名称、参数和延迟
        planned.append({"call_id": f'{task["task_id"]}-c{index}', "tool": tool, "argument": argument, "latency": latency})  # 为每个动作分配唯一且稳定的 call_id
    return planned  # 返回可跨进程传递的调用计划
def execute_envelope(call):  # 执行一个带 call_id 的工具调用
    value = tool_data[(call["tool"], call["argument"])]  # 从确定性工具数据源读取结果
    return {"call_id": call["call_id"], "source_tool": call["tool"], "value": value, "latency": call["latency"]}  # 原样回传调用标识和结果来源
planned_example = plan_calls(tasks[0])  # 为北京天气与雨伞任务生成调用计划
completion_example = sorted((execute_envelope(call) for call in planned_example), key=lambda result: result["latency"])  # 模拟两个结果乱序到达
print("计划顺序：")  # 输出计划阶段的中间量标题
pprint(planned_example, sort_dicts=False)  # 展示每个动作的稳定 call_id
print("实际完成顺序：")  # 输出执行阶段的中间量标题
pprint(completion_example, sort_dicts=False)  # 展示乱序返回时 call_id 仍然跟随结果

计划顺序：
[{'call_id': 'A01-c1', 'tool': 'weather', 'argument': '北京', 'latency': 80},
 {'call_id': 'A01-c2', 'tool': 'inventory', 'argument': '雨伞', 'latency': 20}]
实际完成顺序：
[{'call_id': 'A01-c2',
  'source_tool': 'inventory',
  'value': '库存 18',
  'latency': 20},
 {'call_id': 'A01-c1', 'source_tool': 'weather', 'value': '小雨', 'latency': 80}]


## 4. keyed join：按 call_id 校验并回填计划槽位

编排器先检查未知 ID 和重复 ID，再建立结果字典；最后仍按原计划顺序输出，这样答案结构稳定、执行可以并行。缺失 ID 不应被猜测，而要进入超时或部分失败分支。

In [4]:
def join_by_call_id(planned, completed):  # 按稳定标识关联乱序工具结果
    expected_ids = {call["call_id"] for call in planned}  # 收集本次任务允许出现的调用标识
    result_by_id = {}  # 初始化已完成结果索引
    for result in completed:  # 逐个处理任意顺序到达的结果信封
        if result["call_id"] not in expected_ids:  # 检查结果是否属于当前任务
            raise ValueError(f'unknown_call_id:{result["call_id"]}')  # 拒绝把其他任务结果串入当前答案
        if result["call_id"] in result_by_id:  # 检查同一调用是否被重复提交
            raise ValueError(f'duplicate_call_id:{result["call_id"]}')  # 拒绝重复结果覆盖首次提交
        result_by_id[result["call_id"]] = result  # 按 call_id 保存首次有效结果
    missing_ids = expected_ids - set(result_by_id)  # 计算尚未返回的调用集合
    if missing_ids:  # 检查并行调用是否已经全部完成
        raise ValueError(f'missing_call_ids:{sorted(missing_ids)}')  # 将缺失调用交给超时或重试策略
    joined = [{"call_id": call["call_id"], "tool": call["tool"], "value": result_by_id[call["call_id"]]["value"], "source_tool": result_by_id[call["call_id"]]["source_tool"]} for call in planned]  # 按原计划顺序回填正确结果
    return joined  # 返回稳定且来源可核验的工具结果列表
joined_example = join_by_call_id(planned_example, completion_example)  # 对示例任务执行正确的 keyed join
print("按 call_id 恢复后的计划顺序：")  # 输出关联算法结果标题
pprint(joined_example, sort_dicts=False)  # 展示每个工具槽位拿到自己的真实结果

按 call_id 恢复后的计划顺序：
[{'call_id': 'A01-c1',
  'tool': 'weather',
  'value': '小雨',
  'source_tool': 'weather'},
 {'call_id': 'A01-c2',
  'tool': 'inventory',
  'value': '库存 18',
  'source_tool': 'inventory'}]


## 5. 结果解读：逐任务比较串线数与最终回答

正确结果不仅检查 ID，还检查计划工具与 source_tool 相同。为了贴近 Agent 产物，表格把两个来源组合成最终中文证据串；学习者可以直接看出天气没有再被库存数字替代。

In [5]:
result_rows = []  # 收集六个任务的最终一致性结果
all_joined = {}  # 保存每个任务的已关联结果供回归检查
for task, baseline in zip(tasks, baseline_rows):  # 对齐同一任务的错误基线与正确方案
    planned = plan_calls(task)  # 生成当前任务的稳定调用计划
    completed = sorted((execute_envelope(call) for call in planned), key=lambda result: result["latency"])  # 模拟并行结果按延迟乱序到达
    joined = join_by_call_id(planned, completed)  # 使用 call_id 恢复计划语义
    all_joined[task["task_id"]] = joined  # 保存关联结果便于后续验证
    evidence = "；".join(f'{item["tool"]}={item["value"]}' for item in joined)  # 把正确工具证据组装成人类可读回答
    consistent = all(item["tool"] == item["source_tool"] for item in joined)  # 检查每个结果都回到原调用槽位
    result_rows.append({"任务": task["task_id"], "基线串线数": baseline["串线数"], "正确证据": evidence, "ID关联一致": consistent})  # 保存逐任务对照结果
print("并行工具调用逐样本结果：")  # 输出结果解读标题
pprint(result_rows, sort_dicts=False)  # 展示六个真实任务的串线修复效果

并行工具调用逐样本结果：
[{'任务': 'A01',
  '基线串线数': 2,
  '正确证据': 'weather=小雨；inventory=库存 18',
  'ID关联一致': True},
 {'任务': 'A02',
  '基线串线数': 2,
  '正确证据': 'weather=33℃；inventory=库存 4',
  'ID关联一致': True},
 {'任务': 'A03', '基线串线数': 2, '正确证据': 'payment=已支付；shipment=待出库', 'ID关联一致': True},
 {'任务': 'A04', '基线串线数': 2, '正确证据': 'risk=低风险；credit=额度 5000', 'ID关联一致': True},
 {'任务': 'A05', '基线串线数': 2, '正确证据': 'weather=晴；room=B201 空闲', 'ID关联一致': True},
 {'任务': 'A06',
  '基线串线数': 2,
  '正确证据': 'permission=允许读取；version=v12 最新',
  'ID关联一致': True}]


## 6. 失败案例与修正：重复、未知和缺失 call_id

重试、消息重复投递或跨任务队列污染都会破坏结果集合。下面分别注入一个重复结果、一个陌生结果和一个缺失结果，确认状态机明确拒绝，而不是覆盖字典或凭完成顺序猜测。生产修正通常是：call_id 作为幂等键、首次提交获胜、未知 ID 隔离、缺失 ID 到期转超时。

In [6]:
failure_messages = []  # 收集三类协议破坏的明确错误码
duplicate_results = completion_example + [completion_example[0]]  # 注入同一个 call_id 的重复投递
unknown_result = {"call_id": "OTHER-c9", "source_tool": "weather", "value": "污染结果", "latency": 1}  # 构造来自其他任务的陌生结果
failure_inputs = [duplicate_results, completion_example + [unknown_result], completion_example[:1]]  # 组合重复、未知和缺失三类失败输入
for bad_results in failure_inputs:  # 逐类运行结果集合校验
    try:  # 尝试把协议不合法的结果关联回计划
        join_by_call_id(planned_example, bad_results)  # 执行包含严格 ID 校验的 keyed join
    except ValueError as error:  # 捕获可路由到重试或隔离队列的错误
        failure_messages.append(str(error))  # 保存明确失败原因供观察与告警
print("失败注入与状态机响应：")  # 输出失败案例标题
pprint(failure_messages)  # 展示重复、未知和缺失 ID 都没有被静默吞掉

失败注入与状态机响应：
['duplicate_call_id:A01-c2',
 'unknown_call_id:OTHER-c9',
 "missing_call_ids:['A01-c1']"]


## 7. 生产差距与最小回归检查

真实工具可能产生不可逆副作用，所以 call_id 还应下沉为支付、发信等工具的幂等键。编排器需要持久化 planned/running/succeeded/failed 状态，支持取消、超时、重试预算和跨进程 trace；参数与返回值都要做 schema 校验和敏感字段脱敏。最后的断言只验证本实验已经展示的顺序错配、稳定关联和异常分支。

In [7]:
assert len(tasks) >= 5  # 确认真实并行任务数量满足逐样本教学要求
assert all(row["串线数"] == 2 for row in baseline_rows)  # 确认按完成顺序关联真实造成双结果串线
assert all(row["ID关联一致"] for row in result_rows)  # 确认 call_id 方案在所有任务上保持来源一致
assert len({call["call_id"] for call in planned_example}) == len(planned_example)  # 确认规划阶段生成的调用标识唯一
assert failure_messages[0].startswith("duplicate_call_id")  # 确认重复结果进入显式拒绝分支
assert failure_messages[1].startswith("unknown_call_id")  # 确认跨任务污染结果被隔离
assert failure_messages[2].startswith("missing_call_ids")  # 确认缺失结果不会按顺序猜测补位
print("回归检查通过：乱序关联、稳定 call_id 与三类异常门禁均已验证。")  # 输出最终验收结论

回归检查通过：乱序关联、稳定 call_id 与三类异常门禁均已验证。
